# 3.1.2 矩阵分解：BiasMF

> 阅读版与 Web 应用内容一致；实验数值来自本 Notebook 的已执行输出。

## Goal

单独掌握低秩推荐：理解偏置与 embedding，使用 PyTorch 训练 BiasMF，并分别完成评分预测和全库 Top-K 推理。

## Setup

本 Notebook 的默认真实数据是 **GroupLens MovieLens latest-small：经典评分与邻域任务**。`smoke` 档读取仓库内可审计的确定性切片，`full` 档扩大到官方完整文件；两档都不制造交互、曝光、标签或行为序列。切片规则、源地址、哈希与许可记录在 `data/README.md` 及对应数据目录。

**主要资料：** [Matrix Factorization Techniques for Recommender Systems](https://datajobs.com/data-science-repo/Recommender-Systems-[Netflix].pdf)

In [ ]:
from pathlib import Path
import os, sys, json
import torch
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACT_ROOT = Path(os.environ.get("RECSYS_ARTIFACT_ROOT", PROJECT_ROOT)).expanduser().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("RECSYS_PROFILE", "full")
PROFILE = os.environ["RECSYS_PROFILE"]
CUDA_AVAILABLE = torch.cuda.is_available()
from recsys_lab.data import (load_movielens, movielens_provenance, load_amazon_2023,
                             amazon_provenance, load_kuairand, kuairand_provenance,
                             load_amazon_2018, amazon_2018_provenance,
                             load_movielens_1m, movielens_1m_provenance,
                             load_mind_amazon_books, mind_amazon_provenance,
                             load_census_income, census_income_provenance)
DATASET_KEY = "movielens"
if DATASET_KEY == "movielens":
    real_ratings, real_movies = load_movielens()
    real_interactions = real_ratings
    REAL_DATASET = movielens_provenance(real_ratings)
elif DATASET_KEY == "amazon-books":
    real_ratings = load_amazon_2018("Books") if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = amazon_2018_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "mind-amazon-books":
    real_ratings = load_mind_amazon_books() if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = mind_amazon_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "amazon-electronics":
    real_ratings = load_amazon_2018("Electronics") if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = amazon_2018_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "movielens-1m":
    real_ratings = load_movielens_1m() if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = movielens_1m_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "census-income" and PROFILE == "full":
    census_train_x, census_train_y, census_test_x, census_test_y = load_census_income()
    real_interactions, real_movies, real_ratings = census_train_x, None, census_train_x
    REAL_DATASET = census_income_provenance()
elif DATASET_KEY == "amazon-2023":
    real_ratings = load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = amazon_provenance(real_ratings)
else:
    real_interactions, real_movies = load_kuairand()
    real_ratings = real_interactions
    REAL_DATASET = kuairand_provenance(real_interactions)
print({"profile": PROFILE, "project_root": str(PROJECT_ROOT), "artifact_root": str(ARTIFACT_ROOT), "real_dataset": REAL_DATASET,
       "cuda_available": CUDA_AVAILABLE,
       "cuda_device": torch.cuda.get_device_name(0) if CUDA_AVAILABLE else None})
assert REAL_DATASET["randomly_fabricated_rows"] == 0

## 来源论文与阅读提示

**Koren, Bell & Volinsky (2009)** 总结了 Netflix Prize 时代的矩阵分解实践。论文最值得关注的不是“做一次 SVD”，而是：只在观察集合 $\Omega$ 上优化、显式建模全局/用户/物品偏置，并用正则化控制 user/item 隐向量。本文的 BiasMF 正对应这一结构。

### 公式怎么读（直觉版）

若矩阵或点积陌生，请先阅读 **3.0 推荐算法数学基础**。MF 可以想成给每位用户和每部电影各发一张“兴趣坐标卡”：例如 `[动作偏好, 喜剧偏好]`。两张卡逐项相乘再相加，方向越一致，预测喜欢程度越高。$R\approx PQ^\top$ 只是说：用“用户坐标表 × 物品坐标表”近似原本巨大且稀疏的评分表。

## Steps

## 1. 数据集与评测协议

### 1.1 MovieLens 是什么？

[MovieLens](https://grouplens.org/datasets/movielens/) 是推荐系统最常用的公开教学数据之一。MovieLens 100K 包含用户对电影的 1–5 星评分，核心字段为：

- `user_id`：用户标识；
- `item_id`：电影标识；
- `rating`：显式偏好强度；
- `timestamp`：行为发生时间；
- 电影标题、类型等 metadata。

本 Notebook 使用仓库内固定版本的 **MovieLens latest-small 真实评分**。为控制 CPU 时间，只确定性选取活跃用户和高频电影；所有行仍来自 GroupLens 原始文件，不制造评分或时间戳。这里的数值用于教学回归，不作为统一公开 benchmark 成绩。

### 1.2 为什么按时间切分？

真实推荐只能用过去预测未来。我们把每个用户最后一次行为作为测试目标、其余行为作为训练历史。随机切分会让“未来行为”进入训练集，从而高估效果。

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_squared_error, roc_auc_score, log_loss

SEED = 2026
torch.manual_seed(SEED)
from recsys_lab.data import load_movielens, leave_last_out, movielens_provenance

ratings, movies = load_movielens(max_users=48, max_items=360, min_user_events=12)
train_ratings, test_ratings = leave_last_out(ratings)
n_users, n_items = ratings.user_id.nunique(), ratings.item_id.nunique()

print({
    "rows": len(ratings), "users": ratings.user_id.nunique(), "items": ratings.item_id.nunique(),
    "train_rows": len(train_ratings), "test_rows": len(test_ratings),
    "sparsity": round(1 - len(train_ratings) / (n_users * n_items), 3),
    "source": movielens_provenance(ratings)["source"], "fabricated_rows": 0,
})
display(ratings[["userId", "movieId", "rating", "timestamp", "title", "genres"]].head(8))

### 1.3 两组任务、四类指标

1. **Top-K 推荐**：对每位用户生成 K 个未见物品，检查测试物品是否命中。使用 HitRate@K、Recall@K、Coverage。
2. **评分预测**：预测 1–5 星，使用 RMSE。RMSE 越低越好，但它不直接等价于 Top-K 体验。
3. **点击率预估**：预测曝光后是否点击，使用 AUC 与 LogLoss。AUC 衡量排序，LogLoss 同时惩罚错误且过度自信的概率。

下面先定义几个透明、可复用的评测函数。

In [ ]:
def topk_items(score_matrix, seen_matrix, k=10):
    scores = score_matrix.copy()
    scores[seen_matrix > 0] = -np.inf
    return np.argsort(-scores, axis=1)[:, :k]

def hit_rate_at_k(topk, targets):
    return float(np.mean([target in row for row, target in zip(topk, targets)]))

def catalog_coverage(topk, catalog_size):
    return float(len(np.unique(topk)) / catalog_size)

test_targets = test_ratings.sort_values("user_id").item_id.to_numpy()

In [ ]:
train_matrix = np.zeros((n_users, n_items), dtype=np.float32)
for row in train_ratings.itertuples():
    train_matrix[row.user_id, row.item_id] = float(row.rating >= 3)
print({'interaction_matrix': train_matrix.shape, 'observed_positive': int(train_matrix.sum())})

---

## 3. 矩阵分解（MF）：把用户与物品投影到同一隐空间

### 3.1 从相似度到隐语义

CF 依赖显式邻域；MF 假设评分矩阵近似低秩：一张 $|U|\times|I|$ 的大表，可以近似写成“用户坐标表 $P\in\mathbb R^{|U|\times d}$ 与物品坐标表 $Q\in\mathbb R^{|I|\times d}$ 的乘积”，其中 $d\ll\min(|U|,|I|)$。每个用户用向量 $p_u\in\mathbb R^d$（$P$ 的一行）表示，每个物品用 $q_i\in\mathbb R^d$ 表示，内积 $p_u^\top q_i=\sum_{f=1}^{d}p_{uf}q_{if}$ 代表匹配程度。某一维可能与“动作片偏好”相关，但隐维度是学出来的，通常不可直接解释。

### 3.2 数学：带偏置的评分预测

**通用先修：** [3.0.2 低秩分解与形状](/notebooks/3_0_2_linear_algebra#low-rank-attention) · [3.0.3 偏导与梯度](/notebooks/3_0_3_calculus#derivative-gradient) · [3.0.6 SGD](/notebooks/3_0_6_optimization#sgd) · [3.0.6 L2 与过拟合](/notebooks/3_0_6_optimization#regularization)<br>
**本论文新增数学：** 只在观察集合 $\Omega$ 上拟合 BiasMF，并把评分误差逐步传给用户向量、物品向量与两类偏置。

$$
\hat r_{ui}=\mu+b_u+b_i+p_u^\top q_i
$$

四个符号各司其职：$\mu$ 是全体评分的均值（“大家平均打几颗星”）；$b_u$ 是用户偏置（“这个人手松还是手紧”）；$b_i$ 是物品偏置（“这部作品是否普遍受欢迎”）；$p_u^\top q_i$ 才是个性化部分。

**手算一遍（借用 Koren 论文的数字）。** 设全体均值 $\mu=3.7$ 星；《Titanic》普遍被认为好于平均，$b_i=+0.5$；Joe 打分偏严，$b_u=-0.3$。不做任何个性化时预测为 $3.7+0.5-0.3=3.9$。再设 $d=2$，Joe 的坐标 $p_u=[0.6,-0.2]$（偏动作、不偏浪漫），《Titanic》的坐标 $q_i=[0.5,0.1]$，内积 $=0.6\times0.5+(-0.2)\times0.1=0.28$，最终 $\hat r_{ui}=3.9+0.28=4.18$。

训练时只在已观察评分集合 $\Omega$ 上最小化正则化平方误差。为让单条梯度写得整齐，把每项乘 $\tfrac12$（这只会整体缩放梯度，可由学习率吸收）：

$$
\mathcal L=\frac12\sum_{(u,i)\in\Omega}(r_{ui}-\hat r_{ui})^2
+\frac\lambda2(\|p_u\|_2^2+\|q_i\|_2^2+b_u^2+b_i^2)
$$

$\Omega$ 是“矩阵里有数字的格子”，未观察的格子不参与（它们是缺失，不是 0 分）；$\lambda$ 控制正则强度。对一条评分先缓存旧的 $p_u,q_i$，再按四步算：

1. 预测并求误差：$e_{ui}=r_{ui}-(\mu+b_u+b_i+p_u^\top q_i)$；
2. 求偏导：$\partial L/\partial p_u=-e_{ui}q_i+\lambda p_u$，$\partial L/\partial q_i=-e_{ui}p_u+\lambda q_i$；
3. 偏置偏导：$\partial L/\partial b_u=-e_{ui}+\lambda b_u$，$\partial L/\partial b_i=-e_{ui}+\lambda b_i$；
4. 减去梯度：$p_u\leftarrow p_u+\eta(e_{ui}q_i-\lambda p_u)$，$q_i\leftarrow q_i+\eta(e_{ui}p_u-\lambda q_i)$，$b_u\leftarrow b_u+\eta(e_{ui}-\lambda b_u)$，$b_i$ 同理。

若真实评分高于预测，$e_{ui}>0$，两张坐标卡会被推得更匹配，偏置也会上调；L2 项同时把参数往 0 拉。这里的“低秩”边界也要说清：只有交互矩阵 $PQ^\top$ 的秩至多为 $d$，偏置是额外的行/列平移；模型只近似观察评分，不会恢复唯一的“真实兴趣”。$d$ 太小会欠拟合，太大又可能记住稀疏评分。

### 3.3 训练：用 PyTorch 实现 BiasMF

In [ ]:
class BiasMF(torch.nn.Module):
    def __init__(self, n_users, n_items, embedding_dim=12, global_mean=3.0):
        super().__init__()
        self.user_embedding = torch.nn.Embedding(n_users, embedding_dim)
        self.item_embedding = torch.nn.Embedding(n_items, embedding_dim)
        self.user_bias = torch.nn.Embedding(n_users, 1)
        self.item_bias = torch.nn.Embedding(n_items, 1)
        self.register_buffer("global_mean", torch.tensor(float(global_mean)))
        torch.nn.init.normal_(self.user_embedding.weight, std=0.08)
        torch.nn.init.normal_(self.item_embedding.weight, std=0.08)
        torch.nn.init.zeros_(self.user_bias.weight)
        torch.nn.init.zeros_(self.item_bias.weight)

    def forward(self, user_ids, item_ids):
        interaction = (self.user_embedding(user_ids) * self.item_embedding(item_ids)).sum(dim=1)
        return self.global_mean + self.user_bias(user_ids).squeeze(1) + self.item_bias(item_ids).squeeze(1) + interaction

train_users = torch.tensor(train_ratings.user_id.to_numpy(), dtype=torch.long)
train_items = torch.tensor(train_ratings.item_id.to_numpy(), dtype=torch.long)
train_targets = torch.tensor(train_ratings.rating.to_numpy(), dtype=torch.float32)
test_users = torch.tensor(test_ratings.user_id.to_numpy(), dtype=torch.long)
test_items = torch.tensor(test_ratings.item_id.to_numpy(), dtype=torch.long)
test_targets_rating = test_ratings.rating.to_numpy()

mf_model = BiasMF(n_users, n_items, embedding_dim=12, global_mean=train_targets.mean())
optimizer = torch.optim.Adam(mf_model.parameters(), lr=0.03, weight_decay=1e-4)
loss_curve = []
for epoch in range(160):
    prediction = mf_model(train_users, train_items)
    loss = torch.nn.functional.mse_loss(prediction, train_targets)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    loss_curve.append(float(loss.detach()))

print({"first_loss": round(loss_curve[0], 4), "last_loss": round(loss_curve[-1], 4)})

### 3.4 测试与推理：RMSE 和全库 Top-K

In [ ]:
mf_model.eval()
with torch.no_grad():
    test_prediction = mf_model(test_users, test_items).numpy()
    mf_rmse = float(np.sqrt(mean_squared_error(test_targets_rating, test_prediction)))
    all_users = torch.arange(n_users).repeat_interleave(n_items)
    all_items = torch.arange(n_items).repeat(n_users)
    mf_score_matrix = mf_model(all_users, all_items).reshape(n_users, n_items).numpy()

mf_top10 = topk_items(mf_score_matrix, train_matrix, k=10)
mf_hr10 = hit_rate_at_k(mf_top10, test_targets)
print({"MF test RMSE": round(mf_rmse, 4), "MF HR@10": round(mf_hr10, 4)})
print("user 0 MF 推荐:", mf_top10[0].tolist())

### 3.5 结果讨论与边界

训练损失下降只说明模型拟合了已观察评分；测试 RMSE 才反映未见行为的泛化。Top-K 推理时，MF 与双塔相似：物品向量可以预计算，并用 ANN 搜索内积最大项。

**优点**：比邻域法更紧凑；可平滑稀疏共现；向量检索友好；偏置明确。  
**缺点**：只看 ID 时仍无法解决新实体；平方误差把“未观察”当缺失而非负样本；内积表达能力有限。  
**常见升级**：隐式反馈使用 BPR / sampled-softmax；加入时间偏置、内容特征，或发展为 DSSM 双塔。

## Checks

确认训练损失下降、测试 RMSE 有限、推荐结果不包含已见物品。

In [ ]:
assert loss_curve[-1] < loss_curve[0]
assert np.isfinite(mf_rmse) and mf_rmse < 3
assert not set(mf_top10[0]).intersection(np.where(train_matrix[0] > 0)[0])
print('PASS：BiasMF 训练、评分推理和 Top-K 推理均有效。')

In [ ]:
result_dir = ARTIFACT_ROOT / "results" / "chapter_3_1"; result_dir.mkdir(parents=True, exist_ok=True)
payload = {"records": [{"algorithm":"BiasMF", "task":"评分预测", "metric":"RMSE ↓", "value":mf_rmse, "secondary_metric":"HR@10", "secondary_value":mf_hr10, "online_inference":"user/item 向量内积", "source_notebook":"3_1_2_matrix_factorization"}]}
(result_dir / "matrix_factorization.json").write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("已写出章节指标：matrix_factorization.json")

## Next Steps

把平方误差替换为 BPR pairwise loss；比较 embedding 维数与正则强度；再将 item 向量放入 ANN 索引。